# Job Classification Data Test

This notebook loads and inspects the first 100 most recently updated valid job records that include all three AI summaries from `server/cache/scraped_jobs.sqlite`. It is the starting point for later classification by **for good** industry type.

## 1. Set Up Notebook Dependencies

Import the small set of tools needed for SQLite access and tabular inspection.

In [74]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 240)

workspace_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
database_candidates = [
    base / "server" / "cache" / "scraped_jobs.sqlite"
    for base in workspace_candidates
]
DATABASE_PATH = next((path for path in database_candidates if path.exists()), None)

if DATABASE_PATH is None:
    raise FileNotFoundError(
        "Could not find server/cache/scraped_jobs.sqlite from the notebook or kernel working directory."
    )

ENV_PATH = next(
    (base / "server" / ".env" for base in workspace_candidates if (base / "server" / ".env").exists()),
    None,
)
if ENV_PATH is None:
    raise FileNotFoundError(
        "Could not find server/.env from the notebook or kernel working directory."
    )

with ENV_PATH.open(encoding="utf-8") as env_file:
    for line in env_file:
        stripped_line = line.strip()
        if not stripped_line or stripped_line.startswith("#") or "=" not in stripped_line:
            continue
        key, value = stripped_line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("\"'")
        os.environ.setdefault(key, value)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()

print(f"Using database: {DATABASE_PATH.resolve()}")
print(f"Loaded environment file: {ENV_PATH.resolve()}")
print(f"Gemini API key loaded: {bool(GEMINI_API_KEY)}")

Using database: /Users/8s/Documents/code/web/job_finder_super/server/cache/scraped_jobs.sqlite
Loaded environment file: /Users/8s/Documents/code/web/job_finder_super/server/.env
Gemini API key loaded: True


## 2. Connect to `scraped_jobs.sqlite`

Open the database with a context manager so the connection closes cleanly after the connectivity check.

In [75]:
with sqlite3.connect(DATABASE_PATH) as connection:
    connection.execute("SELECT 1")
    table_count = connection.execute(
        "SELECT COUNT(*) FROM sqlite_master WHERE type = 'table'"
    ).fetchone()[0]

print(f"Connected successfully. Available tables: {table_count}")

Connected successfully. Available tables: 12


## 3. Discover Available Tables and Job Columns

Inspect the database metadata before querying. Jobs are stored as JSON payloads in the `scraped_jobs` table.

In [76]:
with sqlite3.connect(DATABASE_PATH) as connection:
    tables_df = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        connection,
    )

    jobs_schema_df = pd.read_sql_query(
        "PRAGMA table_info(scraped_jobs)",
        connection,
    )

display(tables_df)
display(jobs_schema_df)

,name
0,cache_refresh_requests
1,component_cache_state
2,job_audit_cache
3,job_impact_cache
4,job_qol_cache
5,llm_answer_cache
6,location_latlon_cache
7,location_search_cache
8,lost_and_found
9,scraped_employer_cache


,cid,name,type,notnull,dflt_value,pk
0,0,component_name,TEXT,1,None,1
1,1,job_key,TEXT,1,None,2
2,2,payload_json,TEXT,1,None,0
3,3,updated_at_ms,INTEGER,1,None,0


## 4. Query First 100 Complete Jobs

Use `updated_at_ms` and `job_key` for deterministic ordering. Invalid JSON payloads and jobs missing any of the three AI summaries are excluded before applying the 100-row limit.

In [77]:
TOP_JOBS_QUERY = """
SELECT
    component_name,
    job_key,
    updated_at_ms,
    payload_json
FROM scraped_jobs
WHERE json_valid(payload_json)
  AND TRIM(COALESCE(json_extract(payload_json, '$.scrapedEmployer.employeeQualityOfLifeSummary'), '')) <> ''
  AND TRIM(COALESCE(json_extract(payload_json, '$.scrapedEmployer.ai_summary'), '')) <> ''
  AND TRIM(COALESCE(json_extract(payload_json, '$.scrapedEmployer.ai_impact_summary'), '')) <> ''
ORDER BY updated_at_ms DESC, job_key ASC
LIMIT 1000
"""

with sqlite3.connect(DATABASE_PATH) as connection:
    raw_jobs_df = pd.read_sql_query(TOP_JOBS_QUERY, connection)

print(f"Rows returned: {len(raw_jobs_df)}")
raw_jobs_df.head(3)

Rows returned: 1000


,component_name,job_key,updated_at_ms,payload_json
0,Workday,url:https://logitech.wd5.myworkdayjobs.com/job/cork-ireland/early-careers-program-intern_147590,1788980783883,"{""name"":""Early Careers Program Intern"",""company_name"":""Logitech"",""location"":""2 Locations"",""remote"":""Unknown"",""locati..."
1,Workday,url:https://logitech.wd5.myworkdayjobs.com/job/hong-kong/global-product-marketing-manager--mobile---audio-solutions_...,1788980783883,"{""name"":""Global Product Marketing Manager, Mobile & Audio Solutions"",""company_name"":""Logitech"",""location"":""2 Locatio..."
2,Workday,url:https://logitech.wd5.myworkdayjobs.com/job/hsinchu-taiwan/lead-electrical-engineer_147545,1788980783883,"{""name"":""Lead Electrical Engineer"",""company_name"":""Logitech"",""location"":""Hsinchu, Taiwan"",""remote"":""Unknown"",""locati..."


## 5. Load Results into a DataFrame

Parse the JSON payload and keep only the fields needed for the first job-classification data test: description, AI summaries, job title, and company name.

In [78]:
payload_records = []
for row in raw_jobs_df.to_dict(orient="records"):
    payload = json.loads(row["payload_json"])
    employer = payload.get("scrapedEmployer") or {}
    payload_records.append(
        {
            "job_description": payload.get("description", ""),
            "ai_quality_of_life_summary": employer.get("employeeQualityOfLifeSummary", ""),
            "ai_audit_summary": employer.get("ai_summary", ""),
            "ai_impact_summary": employer.get("ai_impact_summary", ""),
            "job_title": payload.get("name", ""),
            "company_name": payload.get("company_name", ""),
        }
    )

jobs_df = pd.DataFrame(
    payload_records,
    columns=[
        "job_description",
        "ai_quality_of_life_summary",
        "ai_audit_summary",
        "ai_impact_summary",
        "job_title",
        "company_name",
    ],
)

print(f"DataFrame shape: {jobs_df.shape}")
print("Columns:", ", ".join(jobs_df.columns))

DataFrame shape: (1000, 6)
Columns: job_description, ai_quality_of_life_summary, ai_audit_summary, ai_impact_summary, job_title, company_name


## 6. Display and Quick-Check the Retrieved Jobs

Render the six requested fields and confirm every selected row has all three AI summaries for later **for good** industry classification.

In [79]:
display(jobs_df)

blank_counts = jobs_df.apply(
    lambda column: column.astype("string").str.strip().eq("").sum()
).sort_values(ascending=False)
print("Blank counts for requested fields:")
display(blank_counts.to_frame("blank_count"))

print("Text length sample:")
display(
    jobs_df.assign(
        description_length=jobs_df["job_description"].astype("string").str.len(),
        qol_summary_length=jobs_df["ai_quality_of_life_summary"].astype("string").str.len(),
        audit_summary_length=jobs_df["ai_audit_summary"].astype("string").str.len(),
        impact_summary_length=jobs_df["ai_impact_summary"].astype("string").str.len(),
    )[
        [
            "job_title",
            "company_name",
            "description_length",
            "qol_summary_length",
            "audit_summary_length",
            "impact_summary_length",
        ]
    ].head(10)
)

,job_description,ai_quality_of_life_summary,ai_audit_summary,ai_impact_summary,job_title,company_name
0,147590,"Logitech generally offers a positive employee experience, highlighted by a strong focus on work-life balance, flexib...","Logitech is a well-established Swiss multinational public company, founded in 1981, specializing in computer periphe...",Logitech's core mission is to extend human potential in work and play through hardware and software. While not a dir...,Early Careers Program Intern,Logitech
1,146141,"Logitech generally offers a positive employee experience, highlighted by a strong focus on work-life balance, flexib...","Logitech is a well-established Swiss multinational public company, founded in 1981, specializing in computer periphe...",Logitech's core mission is to extend human potential in work and play through hardware and software. While not a dir...,"Global Product Marketing Manager, Mobile & Audio Solutions",Logitech
2,147545,"Logitech generally offers a positive employee experience, highlighted by a strong focus on work-life balance, flexib...","Logitech is a well-established Swiss multinational public company, founded in 1981, specializing in computer periphe...",Logitech's core mission is to extend human potential in work and play through hardware and software. While not a dir...,Lead Electrical Engineer,Logitech
3,145823,"Logitech generally offers a positive employee experience, highlighted by a strong focus on work-life balance, flexib...","Logitech is a well-established Swiss multinational public company, founded in 1981, specializing in computer periphe...",Logitech's core mission is to extend human potential in work and play through hardware and software. While not a dir...,Product Compliance Engineer,Logitech
4,147418,"Logitech generally offers a positive employee experience, highlighted by a strong focus on work-life balance, flexib...","Logitech is a well-established Swiss multinational public company, founded in 1981, specializing in computer periphe...",Logitech's core mission is to extend human potential in work and play through hardware and software. While not a dir...,Sr. CMF Engineer,Logitech
...,...,...,...,...,...,...
995,"Description About onsemi At onsemi, we help improve lives every day through innovative silicon and software solution...","Employee reviews on Indeed.com provide a mixed picture of onsemi's workplace quality of life, with average ratings f...","onsemi is a publicly traded American semiconductor supplier, founded in 1999 as a spin-off from Motorola. The compan...","onsemi states a mission to create energy-efficient technologies to make the world greener, safer, inclusive, and con...",Staff Firmware Developer,onsemi
996,"MISSION The Software Engineer 4 carries individual responsibility for the successful analysis, design, and implement...",Employee reviews for OSI Maritime Systems on platforms like Indeed.com indicate a challenging workplace environment....,"OSI Maritime Systems is a private Canadian defense technology company, founded in either 1977 or 1979, specializing ...",OSI Maritime Systems' core business is dedicated to providing advanced integrated navigation and tactical solutions ...,Sr. Software Engineer,OSI Maritime
997,"Veeam is the Data and AI Trust Company, specializing in helping organizations ensure their data and AI are fully und...","Veeam Software offers a comprehensive benefits package that contributes positively to employee quality of life, incl...","Veeam Software is a privately held, enterprise-stage information technology company founded in 2006, specializing in...","Veeam Software's core business is providing data protection, disaster recovery, and data resilience solutions, which...",Senior Customer Success Engineer - Bilingual French,Veeam Software
998,"Overview Microsoft is a company where passionate innovators come to collaborate, envision what can be and take their...","Microsoft generally offers a healthy employee experience characteri

Blank counts for requested fields:


,blank_count
job_description,0
ai_quality_of_life_summary,0
ai_audit_summary,0
ai_impact_summary,0
job_title,0
company_name,0


Text length sample:


,job_title,company_name,description_length,qol_summary_length,audit_summary_length,impact_summary_length
0,Early Careers Program Intern,Logitech,6,1131,1001,1167
1,"Global Product Marketing Manager, Mobile & Audio Solutions",Logitech,6,1131,1001,1167
2,Lead Electrical Engineer,Logitech,6,1131,1001,1167
3,Product Compliance Engineer,Logitech,6,1131,1001,1167
4,Sr. CMF Engineer,Logitech,6,1131,1001,1167
5,"Sr. Manager, Product Marketing",Logitech,6,1131,1001,1167
6,B2B Category and Alliances Manager - Africa,Logitech,6,1131,1001,1167
7,Sr. Territory Sales Specialist,Logitech,6,1131,1001,1167
8,"PRM/ Salesforce Administrator, Partner Experience",Logitech,6,1131,1001,1167
9,Distribution Channel Marketing Manager,Logitech,6,1131,1001,1167


In [ ]:
from typing import Literal

from google import genai
from pydantic import BaseModel, Field


Category = Literal[
    "Health & Well-being",
    "Climate, Sustainability & Animal Welfare",
    "Education & Opportunity",
    "Open Source & Civic Tech",
    "Social Safety & Community",
    "Responsible Tech & Security",
    "Impact Finance & Economics",
    "Human Rights & Global Development",
    "News, Media Integrity & Information Quality",
]


class JobClassification(BaseModel):
    primary_category: Category
    secondary_category: Category | None = Field(
        default=None,
        description="Optional secondary category; omit when not applicable",
    )
    tertiary_category: Category | None = Field(
        default=None,
        description="Optional third category; omit when not applicable",
    )


client = None


def classify_job_cheap(
    company_name: str,
    company_mission: str,
) -> JobClassification:
    """Classify one job with one required and up to two optional categories."""
    global client
    if not GEMINI_API_KEY:
        raise RuntimeError("GEMINI_API_KEY was not loaded from server/.env.")
    if client is None:
        client = genai.Client(api_key=GEMINI_API_KEY)

    prompt = f"""Classify for Job Search for Good.

Categories:
- Health & Well-being
- Climate, Sustainability & Animal Welfare
- Education & Opportunity
- Open Source & Civic Tech
- Social Safety & Community
- Responsible Tech & Security
- Impact Finance & Economics
- Human Rights & Global Development
- News, Media Integrity & Information Quality

Return one required primary category, plus optional secondary and tertiary categories only when useful. Every category must be distinct.

Input:
Company: {company_name}
Mission: {company_mission[:1000]}"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": JobClassification,
        },
    )
    if response.parsed is None:
        raise ValueError("Gemini returned no parsed JobClassification result.")
    return response.parsed


# Opt-in example: this makes one paid API call when explicitly run.
example_job = jobs_df.iloc[415]
print(example_job)
classification = classify_job_cheap(
    company_name=example_job["company_name"],
    company_mission=example_job["ai_impact_summary"],
)
print(classification)

job_description               About the Role Hi, I'm Yan , Head of Engineering at Loop, and I'm hiring a Staff Software Engineer to be a technical...
ai_quality_of_life_summary    Based on available job descriptions and company information, Loop Financial appears to offer a supportive work envir...
ai_audit_summary              Loop Financial is a Toronto-based financial technology (FinTech) company, operating as a Series A startup that provi...
ai_impact_summary             Loop Financial's core mission is to enable Canadian businesses to expand globally by providing efficient and cost-ef...
job_title                                                                                                                     Staff Software Engineer
company_name                                                                                                                          Join Us at Loop
Name: 415, dtype: object
primary_category='Impact Finance & Economics' secondary_category=None terti

In [ ]:
from math import floor

# Editable defaults for Gemini 3.5 Flash-Lite pricing.
# Update these values if the model's current pricing changes.
INPUT_PRICE_PER_MILLION_TOKENS = 0.10
OUTPUT_PRICE_PER_MILLION_TOKENS = 0.40
CHARS_PER_TOKEN = 4.0
ESTIMATED_OUTPUT_TOKENS = 60


def estimate_classification_cost(
    jobs: pd.DataFrame,
    prompt_count: int = 1000,
    input_price_per_million_tokens: float = INPUT_PRICE_PER_MILLION_TOKENS,
    output_price_per_million_tokens: float = OUTPUT_PRICE_PER_MILLION_TOKENS,
    estimated_output_tokens: int = ESTIMATED_OUTPUT_TOKENS,
    chars_per_token: float = CHARS_PER_TOKEN,
) -> dict[str, float | int]:
    """Estimate batch cost and $1 capacity using the loaded jobs as prompt samples."""
    if prompt_count <= 0:
        raise ValueError("prompt_count must be greater than zero")
    if jobs.empty:
        raise ValueError("jobs must contain at least one row")
    if chars_per_token <= 0:
        raise ValueError("chars_per_token must be greater than zero")

    prompt_lengths = [
        len(
            build_classification_prompt(
                company_name=str(row["company_name"]),
                company_mission=str(row["ai_impact_summary"]),
            )
        )
        for _, row in jobs.iterrows()
    ]
    average_prompt_chars = sum(prompt_lengths) / len(prompt_lengths)
    average_input_tokens = average_prompt_chars / chars_per_token
    input_tokens = average_input_tokens * prompt_count
    output_tokens = estimated_output_tokens * prompt_count
    input_cost = input_tokens / 1_000_000 * input_price_per_million_tokens
    output_cost = output_tokens / 1_000_000 * output_price_per_million_tokens
    total_cost = input_cost + output_cost
    cost_per_prompt = total_cost / prompt_count

    return {
        "prompt_count": prompt_count,
        "sample_job_count": len(jobs),
        "average_prompt_chars": round(average_prompt_chars, 1),
        "average_input_tokens": round(average_input_tokens, 1),
        "estimated_output_tokens_per_prompt": estimated_output_tokens,
        "estimated_input_cost": round(input_cost, 6),
        "estimated_output_cost": round(output_cost, 6),
        "estimated_total_cost": round(total_cost, 6),
        "estimated_cost_per_prompt": round(cost_per_prompt, 8),
        "estimated_prompts_for_one_dollar": floor(1 / cost_per_prompt) if cost_per_prompt > 0 else 0,
    }


cost_estimate = estimate_classification_cost(jobs_df)
print(f"Estimated cost for 1,000 prompts: ${cost_estimate['estimated_total_cost']:.4f}")
print(f"Estimated prompts for $1: {cost_estimate['estimated_prompts_for_one_dollar']:,}")
cost_estimate

Estimated cost for 1,000 prompts: $0.0602
Estimated prompts for $1: 16,597


{'prompt_count': 1000,
 'sample_job_count': 1000,
 'average_prompt_chars': 1449.9,
 'average_input_tokens': 362.5,
 'estimated_output_tokens_per_prompt': 60,
 'estimated_input_cost': 0.036248,
 'estimated_output_cost': 0.024,
 'estimated_total_cost': 0.060248,
 'estimated_cost_per_prompt': 6.025e-05,
 'estimated_prompts_for_one_dollar': 16597}